# N-gram Language Models

This notebook builds unigram, bigram, and trigram language models on tokenized 1B benchmark train/test data, using padded sentences and NLTK frequency distributions. It computes perplexity with and without Laplace smoothing for each model on train and test splits, then uses the trained n‑gram models to generate example sentences via greedy decoding, random sampling, and top‑p (nucleus) sampling with different thresholds.



In [36]:
import math
from collections import Counter, defaultdict
from nltk.util import ngrams
from nltk.probability import ConditionalFreqDist, FreqDist

In [37]:
def read_token_lines(filepath):
    with open(filepath, encoding='utf-8') as f:
        return [line.strip().split() for line in f]

train_file = '/Users/cdmstudent/Downloads/1b_benchmark.train.tokens'
test_file  = '/Users/cdmstudent/Downloads/1b_benchmark.test.tokens'

train_sents = read_token_lines(train_file)
test_sents  = read_token_lines(test_file)

In [54]:
word_counts = Counter(token for sent in train_sents for token in sent)
vocab = {word for word, count in word_counts.items() if count >= 3}
vocab.add('<unk>')
vocab.add('<STOP>')
VOCAB_SIZE = 26602  

def sent_to_vocab(sent, vocab):
    return [w if w in vocab else '<unk>' for w in sent]

train_sents = [sent_to_vocab(sent, vocab) for sent in train_sents]
test_sents  = [sent_to_vocab(sent, vocab) for sent in test_sents]


In [55]:
def pad_sent(sent, n):
    if n == 1: 
        return sent + ['<STOP>']
    elif n == 2: 
        return ['<START>'] + sent + ['<STOP>']
    elif n == 3: 
        return ['<START>', '<START>'] + sent + ['<STOP>']


In [56]:
def unigram(sentences):
    tokens = [w for sent in sentences for w in pad_sent(sent, 1)]
    return FreqDist(tokens)

def bigram(sentences):
    bigram_list = []
    for sent in sentences:
        bigram_list.extend(list(ngrams(pad_sent(sent, 2), 2)))
    return ConditionalFreqDist((b[0], b[1]) for b in bigram_list)

def trigram(sentences):
    trigram_list = []
    for sent in sentences:
        trigram_list.extend(list(ngrams(pad_sent(sent, 3), 3)))
    return ConditionalFreqDist(((t[0], t[1]), t[2]) for t in trigram_list)

In [59]:
def perplexity(sentences, n, model):
    logprob_sum = 0.0
    token_count = 0
    for sent in sentences:
        tokens = pad_sent(sent, n)
        for i in range(n-1, len(tokens)):
            if n == 1:
                prob = model[tokens[i]] / model.N()
            elif n == 2:
                context = tokens[i-1]
                prob = model[context].freq(tokens[i])
            elif n == 3:
                context = (tokens[i-2], tokens[i-1])
                prob = model[context].freq(tokens[i])
            # Handle zero probability: infinite perplexity
            if prob == 0:
                return float('inf')
            logprob_sum += math.log(prob)
            token_count += 1
    return math.exp(-logprob_sum / token_count)

def perplexity_laplace(sentences, n, model, vocab_size):
    logprob_sum = 0.0; token_count = 0
    for sent in sentences:
        tokens = pad_sent(sent, n)
        for i in range(n-1, len(tokens)):
            if n == 1:
                freq = model[tokens[i]]
                total = model.N()
            elif n == 2:
                context = tokens[i-1]
                word = tokens[i]
                freq = model[context][word] if word in model[context] else 0
                total = model[context].N()
            elif n == 3:
                context = (tokens[i-2], tokens[i-1])
                word = tokens[i]
                freq = model[context][word] if word in model[context] else 0
                total = model[context].N()
            prob = (freq + 1) / (total + vocab_size)
            logprob_sum += math.log(prob)
            token_count += 1
    return math.exp(-logprob_sum / token_count)


In [60]:
ngram_fns = {1: unigram, 2: bigram, 3: trigram}
labels = {1: 'Unigram', 2: 'Bigram', 3: 'Trigram'}
splits = {'train': train_sents, 'test': test_sents}

results = []
for n in [1, 2, 3]:
    for split_name, split_data in splits.items():
        model = ngram_fns[n](train_sents if split_name == 'train' else test_sents)
        ppl = perplexity(split_data, n, model)
        ppl_lap = perplexity_laplace(split_data, n, model, VOCAB_SIZE)
        results.append((labels[n], split_name, ppl, ppl_lap))

In [61]:
print("| N-gram | Data Split | Perplexity (No Smoothing) | Perplexity (Laplace) |")
print("|--------|------------|--------------------------|----------------------|")
for row in results:
    print(f"| {row[0]:<7} | {row[1]:<10} | {row[2]:<26.2f} | {row[3]:<22.2f} |")


| N-gram | Data Split | Perplexity (No Smoothing) | Perplexity (Laplace) |
|--------|------------|--------------------------|----------------------|
| Unigram | train      | 976.54                     | 977.51                 |
| Unigram | test       | 849.00                     | 876.98                 |
| Bigram  | train      | 77.07                      | 1442.31                |
| Bigram  | test       | 45.24                      | 2591.43                |
| Trigram | train      | 7.87                       | 6244.42                |
| Trigram | test       | 4.54                       | 8373.14                |


# Part - 2

In [66]:
all_sents = train_sents + test_sents

global_word_counts = Counter(token for sent in all_sents for token in sent)
global_vocab = {w for w, c in global_word_counts.items() if c >= 3}
global_vocab.add('<unk>')
global_vocab.add('<STOP>')
def vocab_convert(sent, vocab):
    return [w if w in vocab else '<unk>' for w in sent]
all_sents = [vocab_convert(sent, global_vocab) for sent in all_sents]


def build_unigram_model(sents):
    tokens = [w for sent in sents for w in sent + ['<STOP>']]
    return FreqDist(tokens)
def build_bigram_model(sents):
    bigrams = []
    for sent in sents:
        bigrams.extend(list(ngrams(['<START>'] + sent + ['<STOP>'], 2)))
    return ConditionalFreqDist((bg[0], bg[1]) for bg in bigrams)
def build_trigram_model(sents):
    trigrams = []
    for sent in sents:
        trigrams.extend(list(ngrams(['<START>', '<START>'] + sent + ['<STOP>'], 3)))
    return ConditionalFreqDist(((tg[0], tg[1]), tg[2]) for tg in trigrams)

ug_model = build_unigram_model(all_sents)
bg_model = build_bigram_model(all_sents)
tg_model = build_trigram_model(all_sents)


def top_p_sample(word_dist, p):
    
    sorted_words = sorted(word_dist, key=lambda w: -word_dist[w])
    sum_p = 0.0
    nucleus = []
    for w in sorted_words:
        sum_p += word_dist[w]
        nucleus.append(w)
        if sum_p >= p:
            break
    return random.choice(nucleus)


def generate_unigram(model, strategy='random', max_len=100, top_p_val = 0.5):
    sent = []
    
    total = model.N()
    word_probs = {w: model[w]/total for w in model}
    for _ in range(max_len):
        if strategy == 'greedy':
            next_word = max(word_probs, key=word_probs.get)
        elif strategy == 'random':
            next_word = random.choices(list(word_probs.keys()), weights=list(word_probs.values()), k=1)[0]
        elif strategy == 'top-p':
            next_word = top_p_sample(word_probs, top_p_val)
        sent.append(next_word)
        if next_word == '<STOP>':
            break
    return ' '.join(sent[:-1])  


def generate_bigram(model, strategy='random', max_len=100, top_p_val = 0.5):
    sent = []
    context = '<START>'
    for _ in range(max_len):
        dist = model[context]
        if len(dist) == 0:
            next_word = '<STOP>' # fallback
        else:
            word_probs = {w: dist[w]/dist.N() for w in dist}
            if strategy == 'greedy':
                next_word = max(word_probs, key=word_probs.get)
            elif strategy == 'random':
                next_word = random.choices(list(word_probs.keys()), weights=list(word_probs.values()), k=1)[0]
            elif strategy == 'top-p':
                next_word = top_p_sample(word_probs, top_p_val)
        if next_word == '<STOP>' or len(sent) >= max_len:
            break
        sent.append(next_word)
        context = next_word
    return ' '.join(sent)


def generate_trigram(model, strategy='random', max_len=100, top_p_val = 0.5):
    sent = []
    context = ('<START>', '<START>')
    for _ in range(max_len):
        dist = model[context]
        if len(dist) == 0:
            next_word = '<STOP>'
        else:
            word_probs = {w: dist[w]/dist.N() for w in dist}
            if strategy == 'greedy':
                next_word = max(word_probs, key=word_probs.get)
            elif strategy == 'random':
                next_word = random.choices(list(word_probs.keys()), weights=list(word_probs.values()), k=1)[0]
            elif strategy == 'top-p':
                next_word = top_p_sample(word_probs, top_p_val)
        if next_word == '<STOP>' or len(sent) >= max_len:
            break
        sent.append(next_word)
        context = (context[1], next_word)
    return ' '.join(sent)


print("Unigram Greedy:", generate_unigram(ug_model, strategy='greedy'))
print("Unigram Random:", generate_unigram(ug_model, strategy='random'))
print("Unigram Top-p:", generate_unigram(ug_model, strategy='top-p', top_p_val=0.8))

print("Bigram Greedy:", generate_bigram(bg_model, strategy='greedy'))
print("Bigram Random:", generate_bigram(bg_model, strategy='random'))
print("Bigram Top-p:", generate_bigram(bg_model, strategy='top-p', top_p_val=0.8))

print("Trigram Greedy:", generate_trigram(tg_model, strategy='greedy'))
print("Trigram Random:", generate_trigram(tg_model, strategy='random'))
print("Trigram Top-p:", generate_trigram(tg_model, strategy='top-p', top_p_val=0.8))


Unigram Greedy: the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the the
Unigram Random: Penney girl <unk> central last And victory
Unigram Top-p: Chris field flights asked accused Company option issues Capital who Chinese local half Italy broke apparently suicide senior don needed Her becoming previous sexual militants know computer network pair Group scene having Open used largest goes stay released Times available returned sent age stores Some $ 12 media fees story School were health major ahead paid women sure net promised analysts short Asia survey beginning Olympic makes Iraqi am bring idea Bill wanted gain Williams Russia testing remaining succes

In [67]:
for p in [0.5, 0.7, 0.9]:
    print(f"Bigram Top-p (p={p}):", generate_bigram(bg_model, strategy='top-p', top_p_val=p))
for p in [0.5, 0.7, 0.9]:
    print(f"Trigram Top-p (p={p}):", generate_trigram(tg_model, strategy='top-p', top_p_val=p))


Bigram Top-p (p=0.5): The council members of a long as part of them to their role in his mind the decision , or implied there was working to cut , was unable to one for this morning , including two hours after an art world 's because it would never again to start .
Bigram Top-p (p=0.7): That you don 't like she met that the animals on your children of France or 2.8 million or changes will pay a mile area where you think of climate <unk> say whether this fall is time he can meet and driving a hero was quickly grow over 200 .
Bigram Top-p (p=0.9): Spain and somewhat <unk> ranch as contentious win and weigh fresh highs and picture by Pedroia drove for reporters ahead and profit targets <unk> fabrics .
Trigram Top-p (p=0.5): That , said she and the other day because they have given up on this subject , as he beat in the West , The <unk> uses the House Committee on Energy Independence and Global Warming ? 8 The Big Breakfast in 1992 .
Trigram Top-p (p=0.7): Mr Spector has refused to say tha